# 07 — Observability: Monitoring AI Systems in Production

## Part 1: Theory

### 1.1 Three Pillars of Observability

| Pillar | What | Tools | Use Case |
|--------|------|-------|----------|
| **Metrics** | Numeric time-series | Prometheus, Datadog | Dashboards, alerts, SLOs |
| **Logs** | Structured events | ELK, CloudWatch | Debugging, audit trail |
| **Traces** | Request flow across services | Jaeger, X-Ray | Latency breakdown, dependencies |

### 1.2 Four Golden Signals (Google SRE)

1. **Latency**: Time to serve a request (distinguish success vs error latency)
2. **Traffic**: Requests per second (demand on your system)
3. **Errors**: Rate of failed requests (5xx, timeouts, wrong answers)
4. **Saturation**: How full is your system (CPU, memory, queue depth)

### 1.3 RED vs USE Methods

```
RED (for services):          USE (for resources):
• Rate (requests/sec)        • Utilization (% busy)
• Errors (failed/sec)        • Saturation (queue depth)
• Duration (latency)         • Errors (error count)
```

### 1.4 SLIs, SLOs, SLAs

```
SLI (Indicator): "What we measure"     → P99 latency = 1.2s
SLO (Objective): "What we promise internally" → P99 < 2s, 99.9% of time
SLA (Agreement): "What we promise customers"  → P99 < 3s, or credits issued

Error Budget = 1 - SLO = allowed failures
  99.9% SLO → 0.1% error budget → 43 minutes/month of downtime allowed
```

### 1.5 Prometheus Metric Types

| Type | Use | Example |
|------|-----|--------|
| **Counter** | Monotonically increasing | Total requests, total errors |
| **Gauge** | Can go up/down | Current queue size, temperature |
| **Histogram** | Distribution in buckets | Request latency (P50, P95, P99) |
| **Summary** | Client-side quantiles | Similar to histogram, pre-computed |

### 1.6 AI-Specific Observability

Beyond standard metrics, AI systems need:
- **Token usage** (cost tracking)
- **Retrieval quality** (similarity scores)
- **Generation quality** (faithfulness, relevance)
- **Hallucination rate** (answers not grounded in context)
- **Model latency breakdown** (retrieval vs generation vs total)

### 1.7 Distributed Tracing for RAG

```
Trace: rag_query (total: 1.8s)
├── Span: embed_query (0.05s)
├── Span: vector_search (0.12s)
│   └── Attribute: num_results=5, top_score=0.87
├── Span: rerank (0.15s)
├── Span: llm_generate (1.4s)
│   └── Attribute: model=llama-3.1-8b, tokens_in=450, tokens_out=120
└── Span: response_format (0.08s)
```

---

## Part 2: Implementation

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prometheus_client import Counter, Histogram, Gauge, CollectorRegistry, generate_latest

### Define RAG Metrics

In [ ]:
reg = CollectorRegistry()

# Request metrics
REQUEST_COUNT = Counter("rag_requests_total", "Total queries", ["status", "model"], registry=reg)
REQUEST_LATENCY = Histogram("rag_request_duration_seconds", "E2E latency",
    buckets=[0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0], registry=reg)

# Retrieval metrics
RETRIEVAL_LATENCY = Histogram("rag_retrieval_seconds", "Vector search latency",
    buckets=[0.01, 0.05, 0.1, 0.25, 0.5], registry=reg)
RETRIEVAL_SCORE = Histogram("rag_retrieval_score", "Top similarity score",
    buckets=[0.5, 0.6, 0.7, 0.8, 0.9, 0.95], registry=reg)

# LLM metrics
LLM_LATENCY = Histogram("rag_llm_seconds", "LLM generation latency",
    buckets=[0.5, 1.0, 2.0, 5.0, 10.0], registry=reg)
TOKEN_USAGE = Counter("rag_tokens_total", "Tokens used", ["type"], registry=reg)

# Quality
ANSWER_CONFIDENCE = Gauge("rag_confidence", "Answer confidence", registry=reg)

print("Metrics defined: request, retrieval, LLM, quality")

### Instrumented RAG Handler

In [ ]:
def instrumented_query(question):
    start = time.time()
    try:
        # Retrieval phase
        ret_lat = random.uniform(0.02, 0.15)
        ret_score = random.uniform(0.6, 0.95)
        RETRIEVAL_LATENCY.observe(ret_lat)
        RETRIEVAL_SCORE.observe(ret_score)
        
        # LLM phase
        llm_lat = random.uniform(0.3, 2.5)
        in_tok = random.randint(200, 800)
        out_tok = random.randint(50, 300)
        LLM_LATENCY.observe(llm_lat)
        TOKEN_USAGE.labels(type="input").inc(in_tok)
        TOKEN_USAGE.labels(type="output").inc(out_tok)
        
        # Simulate 5% error rate
        if random.random() < 0.05:
            raise TimeoutError("LLM timeout")
        
        confidence = ret_score * random.uniform(0.8, 1.0)
        ANSWER_CONFIDENCE.set(confidence)
        REQUEST_LATENCY.observe(time.time() - start)
        REQUEST_COUNT.labels(status="success", model="llama-3.1-8b").inc()
        return {"status": "ok", "confidence": confidence}
    except Exception as e:
        REQUEST_COUNT.labels(status="error", model="llama-3.1-8b").inc()
        REQUEST_LATENCY.observe(time.time() - start)
        return {"status": "error", "error": str(e)}

# Simulate traffic
results = [instrumented_query(f"Q{i}") for i in range(200)]
success = sum(1 for r in results if r["status"] == "ok")
print(f"Simulated 200 queries: {success} success, {200-success} errors ({success/200:.1%} availability)")

### Prometheus Metrics Output

In [ ]:
# This is what your /metrics endpoint serves
metrics_text = generate_latest(reg).decode()
print(metrics_text[:1500])
print("... (truncated)")

### SLO Definitions & Error Budget

In [ ]:
slos = {
    "availability": {"target": 0.999, "window_days": 30},
    "latency_p99_seconds": {"target": 3.0, "window_days": 30},
    "retrieval_quality": {"target": 0.70, "window_days": 7},
}

# Calculate error budget
for name, slo in slos.items():
    budget_minutes = (1 - slo["target"]) * slo["window_days"] * 24 * 60
    print(f"{name}: target={slo['target']}, error budget={budget_minutes:.1f} minutes/{slo['window_days']}d")

# Alert rules
alert_rules = """
groups:
  - name: rag_alerts
    rules:
      - alert: HighErrorRate
        expr: rate(rag_requests_total{status="error"}[5m]) / rate(rag_requests_total[5m]) > 0.01
        for: 2m
        labels: {severity: critical}
      - alert: HighLatency
        expr: histogram_quantile(0.99, rate(rag_request_duration_seconds_bucket[5m])) > 3
        for: 5m
        labels: {severity: warning}
      - alert: LowRetrievalQuality
        expr: histogram_quantile(0.5, rate(rag_retrieval_score_bucket[10m])) < 0.6
        for: 10m
        labels: {severity: warning}
"""
print(alert_rules)

### Dashboard Visualization

In [ ]:
np.random.seed(42)
hours = pd.date_range("2024-01-01", periods=168, freq="h")
data = pd.DataFrame({
    "ts": hours,
    "rpm": np.random.poisson(30, 168) + (np.sin(np.arange(168) * 2*np.pi/24) * 10).astype(int),
    "p50_ms": np.random.normal(400, 50, 168),
    "p95_ms": np.random.normal(1200, 200, 168),
    "error_pct": np.clip(np.random.normal(2, 1, 168), 0, 10),
    "retrieval_score": np.random.normal(0.82, 0.05, 168),
})

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(data["ts"], data["rpm"]); axes[0,0].set_title("Throughput (RPM)"); axes[0,0].set_ylabel("req/min")
axes[0,1].plot(data["ts"], data["p50_ms"], label="P50"); axes[0,1].plot(data["ts"], data["p95_ms"], label="P95")
axes[0,1].axhline(3000, color="r", ls="--", label="SLO"); axes[0,1].set_title("Latency"); axes[0,1].legend()
axes[1,0].plot(data["ts"], data["error_pct"]); axes[1,0].axhline(1, color="r", ls="--", label="SLO")
axes[1,0].set_title("Error Rate (%)"); axes[1,0].legend()
axes[1,1].plot(data["ts"], data["retrieval_score"]); axes[1,1].axhline(0.7, color="r", ls="--", label="SLO")
axes[1,1].set_title("Retrieval Quality"); axes[1,1].legend()
plt.suptitle("RAG Observability Dashboard", fontsize=14); plt.tight_layout(); plt.show()

## Part 3: Key Takeaways

1. **Four golden signals** cover 90% of monitoring needs
2. **AI-specific metrics** (retrieval score, token usage) are essential for RAG
3. **SLOs with error budgets** balance reliability with velocity
4. **Distributed tracing** pinpoints where latency lives (retrieval vs LLM)
5. **Alert on symptoms** (high latency) not causes (high CPU)
6. **Dashboards** should answer: Is the system healthy? Where is it slow? Is quality degrading?

### Next → 08_compliance.ipynb